# 05 — Proposed Retrieval: BGE-small + ChromaDB
**Project:** Semantic Book Recommender — IT4142 HUST  
**Input:**
- `data/processed/books_clean.csv`
- `data/eval/test_queries.json` (from notebook 04)

**Output:**
- `data/chroma_db/` — persistent ChromaDB collection
- `reports/evaluation_baselines.json` — updated with semantic P@K scores

Pipeline:
1. Load data
2. Encode descriptions with `BAAI/bge-small-en-v1.5`
3. Build ChromaDB collection (upsert with metadata)
4. Semantic search function
5. Evaluate Precision@5, Precision@10
6. Compare all 3 models side-by-side
7. Demo search with emotion filter

## 0. Setup

> **RAM requirement:**  
> - `BAAI/bge-small-en-v1.5` (33M params) — needs ~500MB RAM, works on CPU  
> - If RAM < 4GB: use `sentence-transformers/all-MiniLM-L6-v2` instead (see cell below)

In [ ]:
# pip install sentence-transformers chromadb
import pandas as pd
import numpy as np
import json, time
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings

DATA_PATH   = Path('data/processed/books_clean.csv')
EVAL_PATH   = Path('data/eval/test_queries.json')
CHROMA_PATH = Path('data/chroma_db')
REPORT_PATH = Path('reports')
CHROMA_PATH.mkdir(parents=True, exist_ok=True)

print('Setup OK')

## 1. Load Data

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f'Books loaded: {len(df):,}')

with open(EVAL_PATH) as f:
    test_queries = json.load(f)
print(f'Test queries: {len(test_queries)}')

df[['isbn13', 'title', 'categories']].head(3)

## 2. Load Embedding Model

In [ ]:
# Primary model: bge-small-en-v1.5 (recommended)
# Fallback:      all-MiniLM-L6-v2  (lighter, ~80MB)
MODEL_NAME = 'BAAI/bge-small-en-v1.5'
# MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'  # uncomment if low RAM

print(f'Loading model: {MODEL_NAME}')
t0 = time.time()
embed_model = SentenceTransformer(MODEL_NAME)
print(f'Loaded in {time.time()-t0:.1f}s')
print(f'Embedding dim: {embed_model.get_sentence_embedding_dimension()}')

## 3. Encode Book Descriptions

> **BGE prefix:** BGE models perform better when queries are prefixed with `"Represent this sentence: "`.  
> For **documents** (descriptions), no prefix needed.  
> For **queries**, we add the prefix in the search function.

In [ ]:
descriptions = df['description'].tolist()

print(f'Encoding {len(descriptions):,} descriptions...')
t0 = time.time()

embeddings = embed_model.encode(
    descriptions,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,   # L2 normalize → cosine sim = dot product
)

elapsed = time.time() - t0
print(f'Encoded in {elapsed:.1f}s  ({elapsed/len(descriptions)*1000:.1f} ms/book)')
print(f'Embeddings shape: {embeddings.shape}')

## 4. Build ChromaDB Collection

In [ ]:
client = chromadb.PersistentClient(
    path=str(CHROMA_PATH),
    settings=Settings(anonymized_telemetry=False),
)

# Delete collection if exists (clean rebuild)
try:
    client.delete_collection('books')
    print('Deleted existing collection')
except Exception:
    pass

collection = client.create_collection(
    name='books',
    metadata={'hnsw:space': 'cosine'},
)
print(f'Created collection: {collection.name}')

In [ ]:
# Prepare metadata — ChromaDB requires scalar values only (no list/dict)
def safe_str(val, default='Unknown'):
    return str(val) if pd.notna(val) else default

def safe_float(val, default=0.0):
    try:
        return float(val)
    except Exception:
        return default

metadatas = [
    {
        'title'          : safe_str(row['title']),
        'authors'        : safe_str(row['authors']),
        'categories'     : safe_str(row['categories']),
        'thumbnail'      : safe_str(row['thumbnail']),
        'average_rating' : safe_float(row['average_rating']),
        'published_year' : int(row['published_year']) if pd.notna(row.get('published_year')) else 0,
        # top_emotions populated later by notebook 06
        'top_emotions'   : '',
    }
    for _, row in df.iterrows()
]

ids = df['isbn13'].astype(str).tolist()

print(f'Prepared {len(metadatas):,} metadata records')

In [ ]:
# Upsert in batches of 500 (ChromaDB recommended batch size)
BATCH_SIZE = 500
n = len(df)
t0 = time.time()

for start in range(0, n, BATCH_SIZE):
    end = min(start + BATCH_SIZE, n)
    collection.upsert(
        ids=ids[start:end],
        embeddings=embeddings[start:end].tolist(),
        documents=descriptions[start:end],
        metadatas=metadatas[start:end],
    )
    print(f'  Upserted [{start}:{end}]')

elapsed = time.time() - t0
print(f'\nDone in {elapsed:.1f}s')
print(f'Collection count: {collection.count():,}')

## 5. Semantic Search Function

In [ ]:
# BGE prefix for queries (improves retrieval quality)
BGE_QUERY_PREFIX = 'Represent this sentence for searching relevant passages: '

def search_semantic(
    query: str,
    top_k: int = 10,
    filter_emotions: list = None,
) -> pd.DataFrame:
    """
    Semantic search using BGE-small + ChromaDB.

    Args:
        query           : natural language query string
        top_k           : number of results to return
        filter_emotions : list of emotion labels to filter by (requires notebook 06 to have run)
    Returns:
        DataFrame with columns: isbn13, title, authors, categories, average_rating, score
    """
    # Encode query with BGE prefix
    q_emb = embed_model.encode(
        [BGE_QUERY_PREFIX + query],
        normalize_embeddings=True,
    )

    # Build optional where filter for ChromaDB
    where = None
    if filter_emotions:
        # top_emotions is stored as comma-separated string
        # ChromaDB where syntax supports $contains for string fields
        if len(filter_emotions) == 1:
            where = {'top_emotions': {'$contains': filter_emotions[0]}}
        else:
            where = {
                '$or': [
                    {'top_emotions': {'$contains': e}}
                    for e in filter_emotions
                ]
            }

    query_kwargs = dict(
        query_embeddings=q_emb.tolist(),
        n_results=top_k,
        include=['metadatas', 'distances', 'documents'],
    )
    if where:
        query_kwargs['where'] = where

    results = collection.query(**query_kwargs)

    # Parse ChromaDB response into DataFrame
    rows = []
    for i, (meta, dist, doc) in enumerate(zip(
        results['metadatas'][0],
        results['distances'][0],
        results['documents'][0],
    )):
        rows.append({
            'isbn13'         : results['ids'][0][i],
            'title'          : meta.get('title', ''),
            'authors'        : meta.get('authors', ''),
            'categories'     : meta.get('categories', ''),
            'average_rating' : meta.get('average_rating', 0.0),
            'top_emotions'   : meta.get('top_emotions', ''),
            'score'          : round(1 - dist, 4),  # cosine distance → similarity
        })

    return pd.DataFrame(rows)


# Smoke test
test_res = search_semantic('a thriller set in Japan with a detective protagonist', top_k=5)
print('Semantic smoke test:')
test_res[['title', 'authors', 'categories', 'score']]

## 6. Evaluate Precision@K — Semantic

In [ ]:
def is_relevant(book_category: str, relevant_categories: list) -> bool:
    if not isinstance(book_category, str):
        return False
    book_cat_lower = book_category.lower()
    return any(
        rc.lower() in book_cat_lower or book_cat_lower in rc.lower()
        for rc in relevant_categories
    )


def precision_at_k(results_df: pd.DataFrame, relevant_categories: list, k: int) -> float:
    top_k = results_df.head(k)
    hits = top_k['categories'].apply(lambda c: is_relevant(c, relevant_categories)).sum()
    return hits / k


print('Evaluating Semantic (BGE-small + ChromaDB)...')
t0 = time.time()

semantic_scores = {5: [], 10: []}
for q in test_queries:
    res = search_semantic(q['query'], top_k=10)
    for k in [5, 10]:
        semantic_scores[k].append(precision_at_k(res, q['relevant_categories'], k))

semantic_eval = {
    'P@5' : round(np.mean(semantic_scores[5]),  4),
    'P@10': round(np.mean(semantic_scores[10]), 4),
}

print(f'Done in {time.time()-t0:.1f}s  →  {semantic_eval}')

## 7. Final Comparison Table

In [ ]:
# Load baseline scores from notebook 04
baseline_report_path = Path('reports/evaluation_baselines.json')

if baseline_report_path.exists():
    with open(baseline_report_path) as f:
        baseline_report = json.load(f)
    tfidf_scores = baseline_report['models']['tfidf']
    bm25_scores  = baseline_report['models']['bm25']
else:
    # Fallback placeholders if notebook 04 wasn't run first
    tfidf_scores = {'P@5': '?', 'P@10': '?'}
    bm25_scores  = {'P@5': '?', 'P@10': '?'}
    print('Warning: evaluation_baselines.json not found — run notebook 04 first')

comparison = pd.DataFrame([
    {'Model': 'TF-IDF + Cosine',      'Type': 'Sparse baseline',  **tfidf_scores},
    {'Model': 'BM25',                  'Type': 'Sparse baseline',  **bm25_scores},
    {'Model': 'BGE-small + ChromaDB', 'Type': '✓ Dense proposed', **semantic_eval},
])

print('=== Final Retrieval Comparison — Precision@K (50 queries) ===')
print(comparison.to_string(index=False))

## 8. Save Updated Evaluation Report

In [ ]:
# Load existing report and update semantic scores
if baseline_report_path.exists():
    with open(baseline_report_path) as f:
        report = json.load(f)
else:
    report = {'num_queries': len(test_queries), 'models': {}}

report['models']['semantic'] = semantic_eval
report['model_name'] = MODEL_NAME

# Save full report
with open(REPORT_PATH / 'evaluation_baselines.json', 'w') as f:
    json.dump(report, f, indent=2)
print('Updated: reports/evaluation_baselines.json')

# Also save a clean summary
summary = {
    'num_queries'   : len(test_queries),
    'embedding_model': MODEL_NAME,
    'results': [
        {'model': 'tfidf',    'type': 'sparse', **tfidf_scores},
        {'model': 'bm25',     'type': 'sparse', **bm25_scores},
        {'model': 'semantic', 'type': 'dense',  **semantic_eval},
    ]
}
with open(REPORT_PATH / 'evaluation.json', 'w') as f:
    json.dump(summary, f, indent=2)
print('Saved: reports/evaluation.json')

## 9. Demo — Semantic vs Keyword Gap

Đây là nơi semantic search tỏa sáng nhất: query dùng **paraphrase** thay vì từ khóa chính xác.

In [ ]:
import pickle, scipy.sparse as sp
from sklearn.metrics.pairwise import cosine_similarity
from rank_bm25 import BM25Okapi

# Load TF-IDF + BM25 from notebook 04 artifacts
MODEL_PATH = Path('models')
try:
    with open(MODEL_PATH / 'tfidf_vectorizer.pkl', 'rb') as f:
        vectorizer = pickle.load(f)
    tfidf_matrix = sp.load_npz(MODEL_PATH / 'tfidf_matrix.npz')
    with open(MODEL_PATH / 'bm25_index.pkl', 'rb') as f:
        bm25 = pickle.load(f)
    baselines_loaded = True
    print('Baseline models loaded')
except FileNotFoundError:
    baselines_loaded = False
    print('Baseline models not found — run notebook 04 first. Skipping comparison demo.')

In [ ]:
if baselines_loaded:
    def search_tfidf(query, top_k=5):
        q_vec = vectorizer.transform([query])
        scores = cosine_similarity(q_vec, tfidf_matrix).flatten()
        top_idx = np.argsort(scores)[::-1][:top_k]
        res = df.iloc[top_idx][['title', 'authors', 'categories']].copy()
        res['score'] = scores[top_idx].round(4)
        return res.reset_index(drop=True)

    def search_bm25(query, top_k=5):
        scores = bm25.get_scores(query.lower().split())
        top_idx = np.argsort(scores)[::-1][:top_k]
        res = df.iloc[top_idx][['title', 'authors', 'categories']].copy()
        res['score'] = scores[top_idx].round(4)
        return res.reset_index(drop=True)

    # Demo: paraphrase query that doesn't share keywords with books
    DEMO_QUERY = 'a heartbreaking tale about loss and the fragility of human connection'
    K = 5

    print(f'Query: "{DEMO_QUERY}"\n')

    print('--- TF-IDF (keyword match) ---')
    display(search_tfidf(DEMO_QUERY, K))

    print('\n--- BM25 (probabilistic keyword) ---')
    display(search_bm25(DEMO_QUERY, K))

    print('\n--- Semantic / BGE-small (dense, understands meaning) ---')
    display(search_semantic(DEMO_QUERY, K)[['title', 'authors', 'categories', 'score']])

## 10. Demo — Emotion Filter

> Requires notebook `06_emotion_detection.ipynb` to have run and populated `top_emotions` in ChromaDB metadata.

In [ ]:
print('Semantic search WITH emotion filter: ["sadness", "fear"]')
print('Query: "a story about war and its consequences"\n')

try:
    res_filtered = search_semantic(
        'a story about war and its consequences',
        top_k=5,
        filter_emotions=['sadness', 'fear'],
    )
    display(res_filtered[['title', 'categories', 'top_emotions', 'score']])
except Exception as e:
    print(f'Filter failed (likely top_emotions not yet populated): {e}')
    print('Run 06_emotion_detection.ipynb then re-run this cell.')

---
## Done ✓

**Artifacts produced:**
- `data/chroma_db/` — persistent ChromaDB with all book embeddings
- `reports/evaluation.json` — full comparison: TF-IDF / BM25 / Semantic
- `reports/evaluation_baselines.json` — updated with semantic scores

**Next steps:**
- `06_emotion_detection.ipynb` → compute `top_emotions` per book, upsert into ChromaDB
- `07_evaluation_comparison.ipynb` → final report with charts